# Andrea Pezzolla & Matteo Maruca
Progetto 2 DataScience, SUPSI, 2026

In [ ]:
# Import
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
# Dataframe
df = pd.read_csv("100097.csv", sep=';')
#FIXME usa tutti i dati come media annuale così si ha una vera percezione

In [ ]:
# TRADUZIONI COLONNE 

# ── Rinomina colonne in italiano ──────────────────────────────
df = df.rename(columns={
    "Timestamp"                         : "timestamp",
    "Messung-ID"                        : "id_postazione",
    "Richtung ID"                       : "id_direzione",
    "Geschwindigkeit"                   : "velocita_kmh",
    "Zeit"                              : "ora",
    "Datum"                             : "data",
    "Datum und Zeit"                    : "data_ora",
    "Messbeginn"                        : "inizio_misurazione",
    "Messende"                          : "fine_misurazione",
    "Zone"                              : "zona_limite",
    "Ort"                               : "localita",
    "Richtung"                          : "direzione",
    "Koordinaten"                       : "coordinate",
    "Übertretungsquote"                 : "tasso_infrazioni_pct",
    "Geschwindigkeit V50"               : "v50",
    "Geschwindigkeit V85"               : "v85",
    "Strasse"                           : "via",
    "Hausnummer"                        : "numero_civico",
    "Fahrzeuge"                         : "n_veicoli",
    "Fahrzeuglänge"                     : "lunghezza_veicolo_m",
    "Kennzahlen pro Mess-Standort"      : "link_statistiche",
    "geo_point_2d"                      : "geo_point",
})

# ── Conversioni di tipo ───────────────────────────────────────
df["timestamp"]           = pd.to_datetime(df["timestamp"], utc=True)
df["inizio_misurazione"]  = pd.to_datetime(df["inizio_misurazione"])
df["fine_misurazione"]    = pd.to_datetime(df["fine_misurazione"])

# Estrai colonne utili da timestamp
df["anno"]         = df["timestamp"].dt.year
df["mese"]         = df["timestamp"].dt.month
df["giorno_sett"]  = df["timestamp"].dt.day_name(locale="it_IT.UTF-8")  # es. "Lunedì"
df["ora_intera"]   = df["timestamp"].dt.hour

# Zona limite come categoria
#df["zona_limite"] = df["zona_limite"].astype(int).astype(str).apply(lambda x: f"Zona {x}")

print("Colonne rinominate:")
print(df.columns.tolist())
print(f"\nDimensioni: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(df.dtypes)

In [ ]:
df["zona_limite"].unique().max()

In [ ]:
# ── ANALISI 1 ────────────────────────────────────────────────
# Distribuzione delle Velocità per Zona Limite
# ────────────────────────────────────────────────────────────

fig1 = px.histogram(
    df,
    x="velocita_kmh",
    color="zona_limite",
    nbins=60,
    barmode="overlay",
    opacity=0.75,
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione delle Velocità per Zona Limite",
    labels={
        "velocita_kmh": "Velocità (km/h)",
        "count":        "Numero misurazioni",
        "zona_limite":  "Zona"
    }
)
fig1.show()

In [ ]:
# ── ANALISI 2 ────────────────────────────────────────────────
# Top 15 postazioni per tasso medio di infrazioni
# ────────────────────────────────────────────────────────────

# Aggiungiamo 'zona_limite' al groupby per conservare il dato
top_infrazioni = (
    df.groupby(["localita", "via", "zona_limite"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .sort_values("tasso_infrazioni_pct", ascending=False)
    .head(15)
)

fig2 = px.bar(
    top_infrazioni,
    x="tasso_infrazioni_pct",
    y="via",
    orientation="h",
    color="tasso_infrazioni_pct",
    # Alternative di colore se non ti piace il Teal: "Purples", "Reds", "Viridis", "Blues", "Sunset"
    color_continuous_scale="Teal", 
    title="<b>Top 15 Postazioni per Tasso di Infrazioni</b>",
    labels={
        "tasso_infrazioni_pct": "Tasso infrazioni (%)",
        "via":                  "Via",
        "localita":             "Località",
        "zona_limite":          "Limite (km/h)"
    },
    # Configurazione dettagliata dell'hover
    hover_data={
        "via": False, # Nasconde la via dal box perché è già sull'asse Y
        "localita": True,
        "zona_limite": True,
        "tasso_infrazioni_pct": ":.2f" # Formatta il numero con 2 decimali (es. 12.34 al posto di 12.34567)
    }
)

fig2.update_layout(
    yaxis={"categoryorder": "total ascending"},
    coloraxis_showscale=False,
    plot_bgcolor="white", # Sfondo bianco per una lettura più pulita
)

# Aggiungiamo la griglia verticale leggera per facilitare la lettura dei valori
fig2.update_xaxes(showgrid=True, gridcolor="#E5E5E5")

fig2.show()

In [ ]:
# ── ANALISI 3 ────────────────────────────────────────────────
# Mappa di calore – Velocità media per giorno e ora
# ────────────────────────────────────────────────────────────

# Ordine corretto dei giorni
ordine_giorni = ["Lunedì", "Martedì", "Mercoledì", "Giovedì", "Venerdì", "Sabato", "Domenica"]

pivot = (
    df.groupby(["giorno_sett", "ora_intera"])["velocita_kmh"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="velocita_kmh")
    .reindex(ordine_giorni)
)

fig3 = px.imshow(
    pivot,
    color_continuous_scale="inferno",
    aspect="auto",
    title="<b>Velocità Media per Giorno della Settimana e Ora</b>",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Velocità media (km/h)"
    }
)

# Aggiornamento dei font e del design generale
fig3.update_layout(
    title=dict(font=dict(size=20)),
    xaxis_title="<b>Ora del giorno</b>",
    yaxis_title="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

# Forza la visualizzazione di tutte le ore (da 0 a 23)
fig3.update_xaxes(
    tickmode="linear", 
    tick0=0, 
    dtick=1,
    showgrid=False # Rimuoviamo la griglia per non sporcare la mappa di calore
)

fig3.update_yaxes(showgrid=False)

# Tooltip personalizzato per una lettura più pulita
fig3.update_traces(
    hovertemplate="<b>%{y}</b> alle ore <b>%{x}:00</b><br>Velocità media: %{z:.2f} km/h<extra></extra>"
)

fig3.show()

In [ ]:
# ── ANALISI 4 ────────────────────────────────────────────────
# Mappa di calore – infrazioni per giorno e ora (variante)
# Stessa struttura ma su tasso_infrazioni_pct
# ────────────────────────────────────────────────────────────

pivot_inf = (
    df.groupby(["giorno_sett", "ora_intera"])["tasso_infrazioni_pct"]
    .mean()
    .reset_index()
    .pivot(index="giorno_sett", columns="ora_intera", values="tasso_infrazioni_pct")
    .reindex(ordine_giorni)
)

fig4 = px.imshow(
    pivot_inf,
    color_continuous_scale="inferno",
    aspect="auto",
    title="Tasso Infrazioni Medio per Giorno e Ora (%)",
    labels={
        "x":     "Ora del giorno",
        "y":     "Giorno",
        "color": "Infrazioni (%)"
    }
)
fig4.update_layout(
    xaxis_title="Ora del giorno",
    yaxis_title=""
)
fig4.show()

In [ ]:
# ── ANALISI 5 ────────────────────────────────────────────────
# Scatter V50 vs V85 per postazione
# Slide 7 – Correlazione tra velocità mediana e 85° percentile
# ────────────────────────────────────────────────────────────

agg_postazioni = (
    df.groupby(["id_postazione", "via", "zona_limite"])
    .agg(
        v50=("v50",             "mean"),
        v85=("v85",             "mean"),
        infrazioni=("tasso_infrazioni_pct", "mean"),
        n_veicoli=("n_veicoli",     "mean")
    )
    .reset_index()
)

fig5 = px.scatter(
    agg_postazioni,
    x="v50",
    y="v85",
    color="zona_limite",
    size="infrazioni",
    hover_data=["via", "infrazioni", "n_veicoli"],
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Correlazione V50 – V85 per Postazione",
    labels={
        "v50":        "V50 – velocità mediana (km/h)",
        "v85":        "V85 – 85° percentile (km/h)",
        "zona_limite":"Zona",
        "infrazioni": "Tasso infrazioni (%)"
    }
)
fig5.show()

In [ ]:
# ── ANALISI 6 ────────────────────────────────────────────────
# Distribuzione velocità per tipo di veicolo (lunghezza)
# Boxplot – velocita_kmh per categoria lunghezza veicolo
# ────────────────────────────────────────────────────────────

# Categorizza la lunghezza del veicolo
bins   = [0, 3.5, 6, 10, 100]
labels = ["Auto (<3.5m)", "Furgone (3.5–6m)", "Camion (6–10m)", "Autoarticolato (>10m)"]

df["cat_veicolo"] = pd.cut(
    df["lunghezza_veicolo_m"],
    bins=bins,
    labels=labels,
    right=True
)

fig6 = px.box(
    df,
    x="cat_veicolo",
    y="velocita_kmh",
    color="zona_limite",
    color_discrete_map={"Zona 30": "#00B4D8", "Zona 50": "#E07B39"},
    title="Distribuzione Velocità per Categoria di Veicolo e Zona",
    labels={
        "cat_veicolo":  "Categoria veicolo",
        "velocita_kmh": "Velocità (km/h)",
        "zona_limite":  "Zona"
    }
)
fig6.show()

In [ ]:
# ── ANALISI 7 ────────────────────────────────────────────────
# Trend mensile – velocità media e infrazioni nel tempo
# Line chart – evoluzione mese per mese
# ────────────────────────────────────────────────────────────

# Aggregazione dati
trend = (
    df.groupby(["anno", "mese"])
    .agg(
        vel_media  = ("velocita_kmh", "mean"),
        infrazioni = ("tasso_infrazioni_pct", "mean")
    )
    .reset_index()
)
 
# FIX: Rinomina le colonne in inglese per farle capire a to_datetime
trend["periodo"] = pd.to_datetime(
    trend[["anno", "mese"]]
    .rename(columns={"anno": "year", "mese": "month"})
    .assign(day=1)
)
 
# Creazione Subplots
fig7 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1, # Leggero spazio in più tra i due grafici
    subplot_titles=("Velocità media (km/h)", "Tasso infrazioni medio (%)")
)

# Traccia 1: Velocità
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], 
        y=trend["vel_media"],
        mode="lines+markers", 
        name="Velocità",
        line=dict(color="#00B4D8", width=3),
        marker=dict(size=8, symbol="circle", line=dict(color="white", width=1)),
        hovertemplate="<b>%{x|%b %Y}</b><br>Velocità media: %{y:.2f} km/h<extra></extra>"
    ),
    row=1, col=1
)

# Traccia 2: Infrazioni
fig7.add_trace(
    go.Scatter(
        x=trend["periodo"], 
        y=trend["infrazioni"],
        mode="lines+markers", 
        name="Infrazioni",
        line=dict(color="#E07B39", width=3),
        marker=dict(size=8, symbol="circle", line=dict(color="white", width=1)),
        hovertemplate="<b>%{x|%b %Y}</b><br>Tasso infrazioni: %{y:.2f}%<extra></extra>"
    ),
    row=2, col=1
)

# Aggiornamento Layout per un look più pulito e professionale
fig7.update_layout(
    title=dict(text="<b>Trend Mensile</b> – Velocità Media e Tasso Infrazioni", font=dict(size=20)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    hovermode="x unified", # Mostra il tooltip per entrambi i grafici contemporaneamente
    height=600 
)

# Formattazione degli assi e della griglia
fig7.update_yaxes(title_text="km/h", row=1, col=1, showgrid=True, gridcolor="#E5E5E5")
fig7.update_yaxes(title_text="%", row=2, col=1, showgrid=True, gridcolor="#E5E5E5")
fig7.update_xaxes(showgrid=True, gridcolor="#E5E5E5", row=1, col=1)
fig7.update_xaxes(title_text="Mese", showgrid=True, gridcolor="#E5E5E5", row=2, col=1)

fig7.show()

In [ ]:
# ── ANALISI 8 ────────────────────────────────────────────────
# Mappa interattiva delle postazioni (Plotly mapbox)
# Slide 8 futura – punti colorati per tasso infrazioni
# ────────────────────────────────────────────────────────────
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)
 
mappa = (
    df.groupby(["id_postazione", "via", "zona_limite", "lat", "lon"])
    .agg(
        infrazioni = ("tasso_infrazioni_pct", "mean"),
        vel_media  = ("velocita_kmh",          "mean"),
        n_veicoli  = ("n_veicoli",             "mean")
    )
    .reset_index()
)

fig8 = px.scatter_map(
    mappa,
    lat="lat",
    lon="lon",
    color="infrazioni",
    size="n_veicoli",
    size_max=20,
    hover_name="via",
    hover_data={"infrazioni": ":.1f", "vel_media": ":.1f", "zona_limite": True},
    color_continuous_scale="inferno",
    zoom=12,
    map_style="carto-positron",
    title="Mappa Postazioni – Tasso di Infrazioni (%)",
    labels={
        "infrazioni": "Infrazioni (%)",
        "vel_media":  "Vel. media (km/h)",
        "n_veicoli":  "Veicoli"
    }
)
fig8.update_layout(
    margin=dict(l=0, r=0, t=40, b=0)
)
fig8.show()

# Grafici Maru

## Grafico scatter geo che mostra l'omogeneità delle registrazioni nel canton basilea

In [ ]:
df[["lat", "lon"]] = (
    df["geo_point"]
    .str.split(",", expand=True)
    .astype(float)
)

#df['zona_limite'] = df['zona_limite'].str.extract(r'(\d+)')[0].astype(int)

df['Superamento'] = df['velocita_kmh'] - df['zona_limite']
idx_max = df.groupby(['lat', 'lon'])['Superamento'].idxmax()
df_peggiori = df.loc[idx_max].reset_index(drop=True)

# Sostituito scatter_mapbox con scatter_map
fig = px.scatter_map(
    df_peggiori,
    lat="lat",
    lon="lon",
    zoom=12,                                 # Livello di zoom ideale per una città/cantone
    hover_name='zona_limite',
    center={"lat": 47.5596, "lon": 7.5886},  # Coordinate del centro di Basilea
    title="Misurazioni di Velocità - superamento limite kmh",
    size="Superamento",
    color="velocita_kmh",
    color_continuous_scale="Reds"
)

fig.update_layout(map_style="carto-positron")

fig.show()

IL grafico mostra il superamento massimo registrato da ogni radar e si vede bene perche con il colore vediamo la velocita a cui andava e invece con la grandezza quanto ha superato il limite.